# Kaggle Maze Crawler Starter Agent

A compact, notebook-first baseline for the Maze Crawler `crawl` environment. Run this notebook on Kaggle to simulate the starter agents, generate `main.py` and `submission.py` from the same source, and verify both files before submitting.


## 1. Crawl Getting Started

This notebook is the source of truth for the starter agent. It stays intentionally small so each behavior is easy to inspect in replay:

1. `agent_v1` moves only the factory, prioritizing northward survival.
2. `agent_v2` adds one scout that collects visible crystals and returns energy to the factory.

Visual strategy notes and Pilkwang's diagrams live in `docs/2_eda_insights.md`, not in this runnable submission notebook.


## 2. Setup

The simulation cells need Kaggle Environments with the `crawl` environment available. Kaggle usually has the package ready, but this cell installs or upgrades it if needed. The next cell centralizes tunable run settings and the replay helper used by both simulations.


In [ ]:
%%capture
!pip install -q "kaggle-environments>=1.29.0"


In [ ]:
from kaggle_environments import make

RANDOM_SEED = 42
RENDER_WIDTH = 800
RENDER_HEIGHT = 800
RUN_SIMULATIONS = True


def run_simulation(agent_fn, label: str):
    """Run one match against a random opponent and render the replay."""
    if not RUN_SIMULATIONS:
        print(f"{label}: simulation skipped")
        return None

    env = make("crawl", configuration={"randomSeed": RANDOM_SEED}, debug=True)
    env.run([agent_fn, "random"])

    print(label)
    for player_id, step in enumerate(env.steps[-1]):
        print(
            f"Player {player_id}: reward={step.reward}, "
            f"status={step.status}"
        )

    return env.render(
        mode="ipython",
        width=RENDER_WIDTH,
        height=RENDER_HEIGHT,
    )


print("kaggle_environments ready")


## 3. Agent Helpers

Maze walls are stored as bitfields in a flat array. The helpers below keep the coordinate math, wall checks, and robot filtering in one place. They also work with either Kaggle's object-style observations or plain dictionaries, which makes local checks easier.


In [ ]:
from typing import Any

DIRS = {
    "NORTH": (0, 1),
    "SOUTH": (0, -1),
    "EAST": (1, 0),
    "WEST": (-1, 0),
}
WALL_BIT = {
    "NORTH": 1,
    "EAST": 2,
    "SOUTH": 4,
    "WEST": 8,
}


def get_value(obj: Any, key: str, default: Any = None) -> Any:
    """Return a field from either a dict-like or object-like value.

    Args:
        obj: Observation/config object from Kaggle or a plain dict.
        key: Field name to read.
        default: Fallback when the field is missing.

    Returns:
        The requested value or `default`.
    """
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def wall_at(obs: Any, config: Any, col: int, row: int) -> int:
    """Return the known wall bitfield at a cell.

    Args:
        obs: Current Maze Crawler observation.
        config: Environment configuration.
        col: Cell column.
        row: Cell row.

    Returns:
        Wall bitfield. Undiscovered cells are treated as open in this starter.
    """
    walls = get_value(obs, "walls", [])
    south_bound = get_value(obs, "southBound", 0)
    width = get_value(config, "width", 0)
    index = (row - south_bound) * width + col

    if 0 <= index < len(walls) and walls[index] != -1:
        return int(walls[index])
    return 0


def has_road(
    obs: Any,
    config: Any,
    col: int,
    row: int,
    direction: str,
) -> bool:
    """Return True when no known wall blocks a direction from a cell."""
    return not (wall_at(obs, config, col, row) & WALL_BIT[direction])


def neighbor(col: int, row: int, direction: str) -> tuple[int, int]:
    """Return the adjacent cell in `direction`."""
    delta_col, delta_row = DIRS[direction]
    return col + delta_col, row + delta_row


def my_robots(obs: Any) -> dict[str, list[int]]:
    """Return robots controlled by the current player."""
    player = get_value(obs, "player", 0)
    robots = get_value(obs, "robots", {}) or {}
    return {
        uid: data
        for uid, data in robots.items()
        if len(data) > 4 and data[4] == player
    }


def my_factory(obs: Any) -> tuple[str | None, list[int] | None]:
    """Return our factory id and data, if visible."""
    for uid, data in my_robots(obs).items():
        if data[0] == 0:
            return uid, data
    return None, None


def parse_cell(key: str) -> tuple[int, int]:
    """Parse a `col,row` map key into integer coordinates."""
    col, row = key.split(",")
    return int(col), int(row)


## 4. Factory Bug Move

The factory's first job is to stay ahead of the scrolling boundary. This baseline moves north whenever possible, jumps over a north wall when the jump cooldown allows it, and otherwise tries a horizontal sidestep.


In [ ]:
def factory_bug_north(
    col: int,
    row: int,
    jump_cd: int,
    obs: Any,
    config: Any,
) -> str:
    """Choose a simple survival action for the factory."""
    if has_road(obs, config, col, row, "NORTH"):
        return "NORTH"
    if jump_cd == 0:
        return "JUMP_NORTH"
    if has_road(obs, config, col, row, "EAST"):
        return "EAST"
    if has_road(obs, config, col, row, "WEST"):
        return "WEST"
    return "IDLE"


def agent_v1(obs: Any, config: Any) -> dict[str, str]:
    """Factory-only baseline for a quick survival simulation."""
    actions = {}
    for uid, data in my_robots(obs).items():
        robot_type, col, row = data[0], data[1], data[2]
        jump_cd = data[6] if len(data) > 6 else 0
        if robot_type == 0:
            actions[uid] = factory_bug_north(col, row, jump_cd, obs, config)
    return actions


## 5. Agent V1 Simulation

Run the factory-only baseline against a random opponent and render the replay. This is the quick sanity check that `crawl` is installed and the basic action dictionary is valid.


In [ ]:
run_simulation(agent_v1, "Agent V1: factory-only baseline")


## 6. Scout Snail Move

The scout uses a tiny memory: it avoids stepping onto either of the last two cells it visited. That is enough to reduce simple back-and-forth loops while keeping the policy easy to inspect.


In [ ]:
SCOUT_RETURN_ENERGY = 75
DEFAULT_SCOUT_COST = 50
_scout_state: dict[str, dict[str, Any]] = {}


def snail_step(
    uid: str,
    col: int,
    row: int,
    target: tuple[int, int],
    obs: Any,
    config: Any,
) -> str:
    """Move greedily toward a target while avoiding short loops."""
    state = _scout_state.setdefault(uid, {"target": None, "tabu": []})
    if state["target"] != target:
        state["target"] = target
        state["tabu"] = []

    target_col, target_row = target
    candidates = []
    for direction in ("NORTH", "SOUTH", "EAST", "WEST"):
        if not has_road(obs, config, col, row, direction):
            continue

        next_col, next_row = neighbor(col, row, direction)
        if (next_col, next_row) in state["tabu"]:
            continue

        distance = abs(next_col - target_col) + abs(next_row - target_row)
        candidates.append((distance, direction))

    if not candidates:
        state["tabu"] = []
        return "IDLE"

    candidates.sort()
    action = candidates[0][1]
    state["tabu"] = (state["tabu"] + [(col, row)])[-2:]
    return action


def scout_action(
    uid: str,
    col: int,
    row: int,
    energy: int,
    factory_pos: tuple[int, int],
    obs: Any,
    config: Any,
) -> str:
    """Collect visible crystals and ferry high energy back to the factory."""
    factory_col, factory_row = factory_pos
    if energy >= SCOUT_RETURN_ENERGY:
        for direction in ("NORTH", "SOUTH", "EAST", "WEST"):
            is_adjacent = neighbor(col, row, direction) == (
                factory_col,
                factory_row,
            )
            if is_adjacent and has_road(obs, config, col, row, direction):
                return f"TRANSFER_{direction}"
        return snail_step(uid, col, row, factory_pos, obs, config)

    crystals = [
        parse_cell(key)
        for key in (get_value(obs, "crystals", {}) or {})
    ]
    if crystals:
        target = min(
            crystals,
            key=lambda cell: abs(cell[0] - col) + abs(cell[1] - row),
        )
    else:
        target = (col, row + 5)

    return snail_step(uid, col, row, target, obs, config)


def agent_v2(obs: Any, config: Any) -> dict[str, str]:
    """Factory bug-move plus one scout for crystal collection."""
    actions = {}
    robots = my_robots(obs)
    _, factory_data = my_factory(obs)
    scout_count = sum(1 for data in robots.values() if data[0] == 1)

    for uid, data in robots.items():
        robot_type, col, row, energy = data[0], data[1], data[2], data[3]
        jump_cd = data[6] if len(data) > 6 else 0
        build_cd = data[7] if len(data) > 7 else 0

        if robot_type == 0:
            scout_cost = get_value(config, "scoutCost", DEFAULT_SCOUT_COST)
            can_build = scout_count == 0 and build_cd == 0
            if can_build and energy >= scout_cost:
                actions[uid] = "BUILD_SCOUT"
            else:
                actions[uid] = factory_bug_north(
                    col,
                    row,
                    jump_cd,
                    obs,
                    config,
                )
        elif robot_type == 1 and factory_data is not None:
            factory_pos = (factory_data[1], factory_data[2])
            actions[uid] = scout_action(
                uid,
                col,
                row,
                energy,
                factory_pos,
                obs,
                config,
            )

    return actions


## 7. Agent V2 Simulation

Run the factory-plus-scout policy against a random opponent and render the replay. This is the main starter simulation before generating submission files.


In [ ]:
_scout_state.clear()
run_simulation(agent_v2, "Agent V2: factory plus scout")


## 8. Generate Submission Files

Kaggle imports `agent` from `main.py`. This cell writes the final agent source to `main.py`; the next cell mirrors that exact source to `submission.py` for tooling that expects the alternate filename. Keep generated files out of git.


In [ ]:
%%writefile main.py
from typing import Any

DIRS = {
    "NORTH": (0, 1),
    "SOUTH": (0, -1),
    "EAST": (1, 0),
    "WEST": (-1, 0),
}
WALL_BIT = {
    "NORTH": 1,
    "EAST": 2,
    "SOUTH": 4,
    "WEST": 8,
}


def get_value(obj: Any, key: str, default: Any = None) -> Any:
    """Return a field from either a dict-like or object-like value.

    Args:
        obj: Observation/config object from Kaggle or a plain dict.
        key: Field name to read.
        default: Fallback when the field is missing.

    Returns:
        The requested value or `default`.
    """
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def wall_at(obs: Any, config: Any, col: int, row: int) -> int:
    """Return the known wall bitfield at a cell.

    Args:
        obs: Current Maze Crawler observation.
        config: Environment configuration.
        col: Cell column.
        row: Cell row.

    Returns:
        Wall bitfield. Undiscovered cells are treated as open in this starter.
    """
    walls = get_value(obs, "walls", [])
    south_bound = get_value(obs, "southBound", 0)
    width = get_value(config, "width", 0)
    index = (row - south_bound) * width + col

    if 0 <= index < len(walls) and walls[index] != -1:
        return int(walls[index])
    return 0


def has_road(
    obs: Any,
    config: Any,
    col: int,
    row: int,
    direction: str,
) -> bool:
    """Return True when no known wall blocks a direction from a cell."""
    return not (wall_at(obs, config, col, row) & WALL_BIT[direction])


def neighbor(col: int, row: int, direction: str) -> tuple[int, int]:
    """Return the adjacent cell in `direction`."""
    delta_col, delta_row = DIRS[direction]
    return col + delta_col, row + delta_row


def my_robots(obs: Any) -> dict[str, list[int]]:
    """Return robots controlled by the current player."""
    player = get_value(obs, "player", 0)
    robots = get_value(obs, "robots", {}) or {}
    return {
        uid: data
        for uid, data in robots.items()
        if len(data) > 4 and data[4] == player
    }


def my_factory(obs: Any) -> tuple[str | None, list[int] | None]:
    """Return our factory id and data, if visible."""
    for uid, data in my_robots(obs).items():
        if data[0] == 0:
            return uid, data
    return None, None


def parse_cell(key: str) -> tuple[int, int]:
    """Parse a `col,row` map key into integer coordinates."""
    col, row = key.split(",")
    return int(col), int(row)

def factory_bug_north(
    col: int,
    row: int,
    jump_cd: int,
    obs: Any,
    config: Any,
) -> str:
    """Choose a simple survival action for the factory."""
    if has_road(obs, config, col, row, "NORTH"):
        return "NORTH"
    if jump_cd == 0:
        return "JUMP_NORTH"
    if has_road(obs, config, col, row, "EAST"):
        return "EAST"
    if has_road(obs, config, col, row, "WEST"):
        return "WEST"
    return "IDLE"


def agent_v1(obs: Any, config: Any) -> dict[str, str]:
    """Factory-only baseline for a quick survival simulation."""
    actions = {}
    for uid, data in my_robots(obs).items():
        robot_type, col, row = data[0], data[1], data[2]
        jump_cd = data[6] if len(data) > 6 else 0
        if robot_type == 0:
            actions[uid] = factory_bug_north(col, row, jump_cd, obs, config)
    return actions

SCOUT_RETURN_ENERGY = 75
DEFAULT_SCOUT_COST = 50
_scout_state: dict[str, dict[str, Any]] = {}


def snail_step(
    uid: str,
    col: int,
    row: int,
    target: tuple[int, int],
    obs: Any,
    config: Any,
) -> str:
    """Move greedily toward a target while avoiding short loops."""
    state = _scout_state.setdefault(uid, {"target": None, "tabu": []})
    if state["target"] != target:
        state["target"] = target
        state["tabu"] = []

    target_col, target_row = target
    candidates = []
    for direction in ("NORTH", "SOUTH", "EAST", "WEST"):
        if not has_road(obs, config, col, row, direction):
            continue

        next_col, next_row = neighbor(col, row, direction)
        if (next_col, next_row) in state["tabu"]:
            continue

        distance = abs(next_col - target_col) + abs(next_row - target_row)
        candidates.append((distance, direction))

    if not candidates:
        state["tabu"] = []
        return "IDLE"

    candidates.sort()
    action = candidates[0][1]
    state["tabu"] = (state["tabu"] + [(col, row)])[-2:]
    return action


def scout_action(
    uid: str,
    col: int,
    row: int,
    energy: int,
    factory_pos: tuple[int, int],
    obs: Any,
    config: Any,
) -> str:
    """Collect visible crystals and ferry high energy back to the factory."""
    factory_col, factory_row = factory_pos
    if energy >= SCOUT_RETURN_ENERGY:
        for direction in ("NORTH", "SOUTH", "EAST", "WEST"):
            is_adjacent = neighbor(col, row, direction) == (
                factory_col,
                factory_row,
            )
            if is_adjacent and has_road(obs, config, col, row, direction):
                return f"TRANSFER_{direction}"
        return snail_step(uid, col, row, factory_pos, obs, config)

    crystals = [
        parse_cell(key)
        for key in (get_value(obs, "crystals", {}) or {})
    ]
    if crystals:
        target = min(
            crystals,
            key=lambda cell: abs(cell[0] - col) + abs(cell[1] - row),
        )
    else:
        target = (col, row + 5)

    return snail_step(uid, col, row, target, obs, config)


def agent_v2(obs: Any, config: Any) -> dict[str, str]:
    """Factory bug-move plus one scout for crystal collection."""
    actions = {}
    robots = my_robots(obs)
    _, factory_data = my_factory(obs)
    scout_count = sum(1 for data in robots.values() if data[0] == 1)

    for uid, data in robots.items():
        robot_type, col, row, energy = data[0], data[1], data[2], data[3]
        jump_cd = data[6] if len(data) > 6 else 0
        build_cd = data[7] if len(data) > 7 else 0

        if robot_type == 0:
            scout_cost = get_value(config, "scoutCost", DEFAULT_SCOUT_COST)
            can_build = scout_count == 0 and build_cd == 0
            if can_build and energy >= scout_cost:
                actions[uid] = "BUILD_SCOUT"
            else:
                actions[uid] = factory_bug_north(
                    col,
                    row,
                    jump_cd,
                    obs,
                    config,
                )
        elif robot_type == 1 and factory_data is not None:
            factory_pos = (factory_data[1], factory_data[2])
            actions[uid] = scout_action(
                uid,
                col,
                row,
                energy,
                factory_pos,
                obs,
                config,
            )

    return actions

def compute_actions(obs: Any, config: Any) -> dict[str, str]:
    """Return the production starter action dictionary."""
    return agent_v2(obs, config)


def idle_actions(obs: Any) -> dict[str, str]:
    """Return safe idle actions for visible owned robots."""
    return {uid: "IDLE" for uid in my_robots(obs)}


def agent(obs: Any, config: Any) -> dict[str, str]:
    """Kaggle-facing entry point."""
    try:
        return compute_actions(obs, config)
    except Exception:
        return idle_actions(obs)


def act(obs: Any, config: Any) -> dict[str, str]:
    """Alias used by some local runners."""
    return agent(obs, config)


__all__ = ["agent", "act", "compute_actions"]


## 9. Verify Generated Files

The generated files should compile and stay byte-for-byte identical. Keep this check close to the write cell so stale submission artifacts are obvious.


In [ ]:
from pathlib import Path
import py_compile

main_path = Path("main.py")
submission_path = Path("submission.py")
submission_path.write_text(
    main_path.read_text(encoding="utf-8"),
    encoding="utf-8",
)

py_compile.compile(str(main_path), doraise=True)
py_compile.compile(str(submission_path), doraise=True)
assert main_path.read_text(encoding="utf-8") == submission_path.read_text(
    encoding="utf-8"
)

print("main.py: OK")
print("submission.py: OK")
print("submission.py sync: OK")


## 10. Submit To The Leaderboard

Run the generation and verification cells, save the Kaggle notebook, then use **Submit to Competition** from the Kaggle notebook panel. Kaggle will use the generated `main.py`; `submission.py` is available for downloads, local checks, and helper tooling.


## 11. Next Improvements

- Add a worker to remove walls in front of the factory with `REMOVE_NORTH`.
- Add miners and `TRANSFORM` them on mining nodes once energy recovery is reliable.
- Split multiple scouts across different crystals instead of sending everyone to the same target.
- Replace snail move with BFS or A* after caching enough wall information.
